## Cluster NIFTY50 companies into different clusters based on the 1-year volatility and returns

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import yfinance as yf

warnings.filterwarnings('ignore')

## Read Dataset

In [2]:
nifty50_stock_tickers = []
pd.read_csv('ind_nifty50list.csv')['Symbol'].apply(lambda x: nifty50_stock_tickers.append(f'{x}.NS'))
nifty50_stock_tickers

['ADANIENT.NS',
 'ADANIPORTS.NS',
 'APOLLOHOSP.NS',
 'ASIANPAINT.NS',
 'AXISBANK.NS',
 'BAJAJ-AUTO.NS',
 'BAJFINANCE.NS',
 'BAJAJFINSV.NS',
 'BPCL.NS',
 'BHARTIARTL.NS',
 'BRITANNIA.NS',
 'CIPLA.NS',
 'COALINDIA.NS',
 'DIVISLAB.NS',
 'DRREDDY.NS',
 'EICHERMOT.NS',
 'GRASIM.NS',
 'HCLTECH.NS',
 'HDFCBANK.NS',
 'HDFCLIFE.NS',
 'HEROMOTOCO.NS',
 'HINDALCO.NS',
 'HINDUNILVR.NS',
 'ICICIBANK.NS',
 'ITC.NS',
 'INDUSINDBK.NS',
 'INFY.NS',
 'JSWSTEEL.NS',
 'KOTAKBANK.NS',
 'LTIM.NS',
 'LT.NS',
 'M&M.NS',
 'MARUTI.NS',
 'NTPC.NS',
 'NESTLEIND.NS',
 'ONGC.NS',
 'POWERGRID.NS',
 'RELIANCE.NS',
 'SBILIFE.NS',
 'SHRIRAMFIN.NS',
 'SBIN.NS',
 'SUNPHARMA.NS',
 'TCS.NS',
 'TATACONSUM.NS',
 'TATAMOTORS.NS',
 'TATASTEEL.NS',
 'TECHM.NS',
 'TITAN.NS',
 'ULTRACEMCO.NS',
 'WIPRO.NS']

## Download the OLCH and volume data for these stocks

In [3]:
raw_data = yf.download(nifty50_stock_tickers, period='1Y',auto_adjust = True)
data = raw_data['Close']
data.info()


[*********************100%%**********************]  50 of 50 completed

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 245 entries, 2023-08-07 to 2024-08-07
Data columns (total 50 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   ADANIENT.NS    245 non-null    float64
 1   ADANIPORTS.NS  245 non-null    float64
 2   APOLLOHOSP.NS  245 non-null    float64
 3   ASIANPAINT.NS  245 non-null    float64
 4   AXISBANK.NS    245 non-null    float64
 5   BAJAJ-AUTO.NS  245 non-null    float64
 6   BAJAJFINSV.NS  245 non-null    float64
 7   BAJFINANCE.NS  245 non-null    float64
 8   BHARTIARTL.NS  245 non-null    float64
 9   BPCL.NS        245 non-null    float64
 10  BRITANNIA.NS   245 non-null    float64
 11  CIPLA.NS       245 non-null    float64
 12  COALINDIA.NS   245 non-null    float64
 13  DIVISLAB.NS    245 non-null    float64
 14  DRREDDY.NS     245 non-null    float64
 15  EICHERMOT.NS   245 non-null    float64
 16  GRASIM.NS      245 non-null    float64
 17  HCLTECH.NS     245 non-null    floa

In [4]:
data.describe()

,ADANIENT.NS,ADANIPORTS.NS,APOLLOHOSP.NS,ASIANPAINT.NS,AXISBANK.NS,BAJAJ-AUTO.NS,BAJAJFINSV.NS,BAJFINANCE.NS,BHARTIARTL.NS,BPCL.NS,...,SHRIRAMFIN.NS,SUNPHARMA.NS,TATACONSUM.NS,TATAMOTORS.NS,TATASTEEL.NS,TCS.NS,TECHM.NS,TITAN.NS,ULTRACEMCO.NS,WIPRO.NS
count,245.000000,245.000000,245.000000,245.000000,245.000000,245.000000,245.000000,245.000000,245.000000,245.000000,...,245.000000,245.000000,245.000000,245.000000,245.000000,245.000000,245.000000,245.000000,245.000000,245.000000
mean,2873.739732,1144.291105,5783.655176,3019.698998,1088.300320,7325.187685,1597.567975,7105.979331,1132.996522,251.789171,...,2254.794510,1369.993236,1037.746928,835.143882,142.273810,3748.376806,1261.602467,3425.165850,9603.537636,458.844324
std,353.808081,263.655323,586.140998,168.250658,89.983989,1822.552112,56.257386,394.792468,200.078735,62.666509,...,343.606684,196.021048,124.643821,163.654114,19.842723,283.183375,101.184003,220.913635,1068.598801,46.699018
min,2148.983398,765.814270,4756.009277,2684.129639,932.330444,4547.094238,1459.229370,6279.725098,850.460571,159.919739,...,1761.223511,1079.007935,826.849365,599.151062,113.512978,3283.738770,1083.479858,2895.126953,8006.817383,377.445038
25%,2505.439697,822.172485,5133.119629,2876.963867,1020.112732,5363.261230,1568.060791,6792.550293,942.050110,174.863724,...,1928.949707,1150.571899,907.096069,653.682800,125.374001,3480.428955,1193.292480,3256.394775,8585.435547,416.212372
50%,3005.388184,1239.233154,5903.450195,2973.034424,1079.566895,7658.159668,1592.895020,7115.549805,1119.511230,283.325012,...,2290.483643,1448.456909,1090.630737,904.109985,137.087997,3804.349121,1243.855713,3411.038574,9730.097656,461.000000
75%,3169.399902,1342.888428,6238.549805,3160.499756,1131.127075,8936.867188,1631.370850,7349.803223,1314.971924,306.600006,...,2452.867432,1535.181885,1135.868774,983.714355,161.006088,3935.489258,1296.529907,3612.579102,9978.972656,491.850006
max,3643.780518,1590.150024,6774.049805,3371.825684,1317.300049,9961.750000,1732.057251,8127.747559,1506.007324,350.049988,...,2992.100098,1734.449951,1252.911987,1161.849976,179.940002,4397.100098,1554.400024,3854.039062,11984.500000,573.200012


In [5]:
data.shape

(245, 50)

In [6]:
data.isna().sum()

ADANIENT.NS      0
ADANIPORTS.NS    0
APOLLOHOSP.NS    0
ASIANPAINT.NS    0
AXISBANK.NS      0
BAJAJ-AUTO.NS    0
BAJAJFINSV.NS    0
BAJFINANCE.NS    0
BHARTIARTL.NS    0
BPCL.NS          0
BRITANNIA.NS     0
CIPLA.NS         0
COALINDIA.NS     0
DIVISLAB.NS      0
DRREDDY.NS       0
EICHERMOT.NS     0
GRASIM.NS        0
HCLTECH.NS       0
HDFCBANK.NS      0
HDFCLIFE.NS      0
HEROMOTOCO.NS    0
HINDALCO.NS      0
HINDUNILVR.NS    0
ICICIBANK.NS     0
INDUSINDBK.NS    0
INFY.NS          0
ITC.NS           0
JSWSTEEL.NS      0
KOTAKBANK.NS     0
LT.NS            0
LTIM.NS          0
M&M.NS           0
MARUTI.NS        0
NESTLEIND.NS     0
NTPC.NS          0
ONGC.NS          0
POWERGRID.NS     0
RELIANCE.NS      0
SBILIFE.NS       0
SBIN.NS          0
SHRIRAMFIN.NS    0
SUNPHARMA.NS     0
TATACONSUM.NS    0
TATAMOTORS.NS    0
TATASTEEL.NS     0
TCS.NS           0
TECHM.NS         0
TITAN.NS         0
ULTRACEMCO.NS    0
WIPRO.NS         0
dtype: int64

## Feature Extraction

In [7]:
stock_df = pd.DataFrame()

stock_df['returns'] = (data.pct_change()+1).prod() - 1
stock_df['volatility'] = data.pct_change().std()*np.sqrt(252)
stock_df

,returns,volatility
ADANIENT.NS,0.249651,0.422495
ADANIPORTS.NS,0.959090,0.413319
APOLLOHOSP.NS,0.346343,0.221800
ASIANPAINT.NS,-0.061644,0.182358
AXISBANK.NS,0.200842,0.242894
BAJAJ-AUTO.NS,1.095694,0.239086
BAJAJFINSV.NS,0.042423,0.209799
BAJFINANCE.NS,-0.063419,0.247007
BHARTIARTL.NS,0.633884,0.216124
BPCL.NS,1.018019,0.346710


In [9]:
stock_df = stock_df.sort_values(by='returns',ascending=False)
stock_df

,returns,volatility
COALINDIA.NS,1.456030,0.350770
BAJAJ-AUTO.NS,1.095694,0.239086
BPCL.NS,1.018019,0.346710
POWERGRID.NS,1.000445,0.308010
ONGC.NS,0.994112,0.360458
NTPC.NS,0.990180,0.316015
ADANIPORTS.NS,0.959090,0.413319
HEROMOTOCO.NS,0.796884,0.256417
M&M.NS,0.768603,0.293070
TATAMOTORS.NS,0.687390,0.274570


In [ ]:
 sns.scatterplot(x='volatility',y='returns',data=stock_df)

## Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaled_features = scaler.fit_transform(stock_df)
stock_df = pd.DataFrame(scaled_features,columns=stock_df.columns)
stock_df

In [ ]:
stock_df = stock_df.describe()
stock_df